In [18]:
from ultralytics import YOLO
import math

 
DETECT = YOLO("yolov8n.pt")
POSE = YOLO("yolov8n-pose.pt")
 
VIDEO = "resources/vid2_short.mp4"   

LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_WRIST, RIGHT_WRIST = 9, 10
NEAR = 0.5
 
 

In [16]:
detect_res = DETECT.predict(source=VIDEO, verbose=False)
pose_res = POSE.predict(source=VIDEO, verbose=False)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
 

In [21]:
def point_to_box(px, py, box):
    x1, y1, x2, y2 = box
    dx = max(x1 - px, 0, px - x2)   # horizontal gap (0 if inside the left-right range)
    dy = max(y1 - py, 0, py - y2)   # vertical gap (0 if inside the top-bottom range)
    return math.hypot(dx, dy)       # straight-line combo of both

In [ ]:

for frame_num, (det, pose) in enumerate(zip(detect_res, pose_res)):
    # Skip frames with no person
    if pose.keypoints is None or len(pose.keypoints.xy) == 0:
        continue
 
    positions = pose.keypoints.xy[0].cpu().numpy()
    shoulder_width = math.dist(positions[LEFT_SHOULDER], positions[RIGHT_SHOULDER])
 
    # Skip if shoulders weren't found (they come back as 0,0) or are too close together
    if shoulder_width < 1:
        continue
 
    wrists = {"left": positions[LEFT_WRIST], "right": positions[RIGHT_WRIST]}
 
    for box, cls in zip(det.boxes.xyxy.cpu().numpy(), det.boxes.cls):
        name = det.names[int(cls)]
        if name == "person":          # the wrist is always inside the person box
            continue
 
        for side, (wx, wy) in wrists.items():
            if wx == 0 and wy == 0:   # this wrist wasn't detected
                continue
            distance = point_to_box(wx, wy, box) / shoulder_width
            print(f"frame {frame_num}: {side} hand to {name} = {distance:.2f}")
            if distance < NEAR:
                print(f"   -> {side} hand is near the {name}")
 

frame 0: left hand to cup = 0.40
   -> left hand is near the cup
frame 0: right hand to cup = 0.38
   -> right hand is near the cup
frame 0: left hand to potted plant = 0.00
   -> left hand is near the potted plant
frame 0: right hand to potted plant = 0.05
   -> right hand is near the potted plant
frame 0: left hand to dining table = 0.00
   -> left hand is near the dining table
frame 0: right hand to dining table = 0.00
   -> right hand is near the dining table
frame 0: left hand to cup = 0.07
   -> left hand is near the cup
frame 0: right hand to cup = 0.72
frame 1: left hand to cup = 0.40
   -> left hand is near the cup
frame 1: right hand to cup = 0.38
   -> right hand is near the cup
frame 1: left hand to potted plant = 0.00
   -> left hand is near the potted plant
frame 1: right hand to potted plant = 0.05
   -> right hand is near the potted plant
frame 1: left hand to dining table = 0.00
   -> left hand is near the dining table
frame 1: right hand to dining table = 0.00
   -> r